<a href="https://colab.research.google.com/github/NoT-Serna/Juan_Serna_FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule will be based on two signals to create the "needs_review" label
1. CTR-vs-Position: Are pages with a low CTR in comparison to its postion a good review candidate?
2. Impression Volume: Should we only review pages that have enough impressions?

RULE: The pages that will need review and therefore recieve the "needs_review" label, are pages that have a lower than expected CTR for their search position and have enough impressions.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

# Read the token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Register the Hugging Face secret
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

# Warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Test the connection
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").show()





FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



In [ ]:
#====== RANKED CSV ==============#
df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""").df()

#Aggreation query by month of April
agg = con.sql(f"""
SELECT
    month,
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')

WHERE month = '2025-04'

GROUP BY
    month,
    client_hash_id,
    content_hash_id
""").df()

#Computing CTR
agg["ctr"] = np.where(
    agg["impressions"] > 0,
    agg["clicks"] / agg["impressions"],
    0
)

# Postion buckets
agg["position_bucket"] = pd.cut(
    agg["avg_position"],
    bins=[0, 3, 5, 10, 20, 100],
    labels =["1-3", "3-5", "5-10", "10-20", "20+"],
    include_lowest=True
)

# FIRST SIGNAL
signal1 = (
    agg
    .groupby("position_bucket")
    .agg(
        avg_ctr = ("ctr", "mean"),
        avg_impressions=("impressions", "mean"),
        n=("content_hash_id", "count")
    )
    .reset_index()
)
signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_989/2182429623.py:47: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("position_bucket")


,position_bucket,avg_ctr,avg_impressions,n
0,1-3,0.019535,501.486239,109
1,3-5,0.018284,2720.286207,290
2,5-10,0.008456,1534.537805,2460
3,10-20,0.006893,668.366950,2826
4,20+,0.002238,317.773290,7353


VEREDCIT= Confirmed

In [ ]:
#SECOND SIGNAL
agg["impression_bucket"] = pd.qcut(
    agg["impressions"],
    q = 4,
    labels = ["Low", "Medium", "High", "Very High"],
    duplicates = "drop"
)

signal2 = (
    agg
    .groupby("impression_bucket")
    .agg(
        avg_ctr = ("ctr", "mean"),
        avg_position=("avg_position", "mean"),
        n=("content_hash_id", "count")
    )
    .reset_index()
)
signal2

/tmp/ipykernel_989/330269181.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("impression_bucket")


,impression_bucket,avg_ctr,avg_position,n
0,Low,0.004803,29.705569,3283
1,Medium,0.004061,33.169546,3251
2,High,0.004908,30.716254,3252
3,Very High,0.005912,23.108411,3260


VEREDICT = Mixed

**My** **rule**


In [13]:
#RULE
import numpy as np
import pandas as pd
from pathlib import Path

# Position buckets
agg["position_bucket"] = pd.cut(
    agg["avg_position"],
    bins=[0, 3, 5, 10, 20, float("inf")],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    include_lowest=True
)

# Expected CTR from the signal analysis
expected_ctr = {
    "1-3": 0.019535,
    "3-5": 0.018284,
    "5-10": 0.008456,
    "10-20": 0.006893,
    "20+": 0.002238
}

agg["expected_ctr"] = (
    agg["position_bucket"]
    .astype(str)
    .map(expected_ctr)
)

# Difference between expected and actual CTR
agg["ctr_gap"] = agg["expected_ctr"] - agg["ctr"]

# Label
agg["needs_review"] = (
    (agg["ctr_gap"] > 0) &
    (agg["impressions"] >= 100)
).astype(int)

# Ranking score
agg["score"] = (
    agg["ctr_gap"] *
    np.log1p(agg["impressions"])
)

# Reason code
agg["reason_code"] = np.where(
    agg["needs_review"] == 1,
    "CTR_GAP",
    "NONE"
)

# Action
agg["action"] = np.where(
    agg["needs_review"] == 1,
    "Improve Title & Meta",
    "No Action"
)

# Ranked queue
queue = (
    agg.sort_values("score", ascending=False)
)

# Save CSV
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_dir / "baseline_action_score.csv", index=False)

print(f"Saved to {output_dir / 'baseline_action_score.csv'}")

Saved to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
queue.head(20)


,month,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,position_bucket,impression_bucket,expected_ctr,ctr_gap,needs_review,score,reason_code,action
6763,2025-04,client_73cda7b4e4f265ea,content_0b4178c50db57c23,23890.0,90.0,2.798314,0.003767,1-3,Very High,0.019535,0.015768,1,0.158959,CTR_GAP,Improve Title & Meta
946,2025-04,client_73cda7b4e4f265ea,content_8989056a93f310f9,3664.0,2.0,4.739496,0.000546,3-5,Very High,0.018284,0.017738,1,0.145570,CTR_GAP,Improve Title & Meta
2825,2025-04,client_9958f0a7ae1df715,content_79bfcf05bc81bf82,12394.0,42.0,3.355194,0.003389,3-5,Very High,0.018284,0.014895,1,0.140389,CTR_GAP,Improve Title & Meta
866,2025-04,client_73cda7b4e4f265ea,content_c8a344d6fbdbb873,6432.0,18.0,4.834496,0.002799,3-5,Very High,0.018284,0.015485,1,0.135795,CTR_GAP,Improve Title & Meta
2518,2025-04,client_9958f0a7ae1df715,content_f3a75d8cf58dd50b,3869.0,8.0,3.267181,0.002068,3-5,Very High,0.018284,0.016216,1,0.133963,CTR_GAP,Improve Title & Meta
5180,2025-04,client_73cda7b4e4f265ea,content_f08cd9083f18f777,2446.0,3.0,4.706492,0.001226,3-5,Very High,0.018284,0.017058,1,0.133093,CTR_GAP,Improve Title & Meta
6871,2025-04,client_73cda7b4e4f265ea,content_3c1f61c9fa1aa75b,3448.0,7.0,4.501050,0.002030,3-5,Very High,0.018284,0.016254,1,0.132401,CTR_GAP,Improve Title & Meta
8418,2025-04,client_9958f0a7ae1df715,content_43e2f39f8ecdf102,3257.0,7.0,4.426595,0.002149,3-5,Very High,0.018284,0.016135,1,0.130512,CTR_GAP,Improve Title & Meta
6819,2025-04,client_73cda7b4e4f265ea,content_2915a57c1fd91871,4877.0,18.0,4.851624,0.003691,3-5,Very High,0.018284,0.014593,1,0.123933,CTR_GAP,Improve Title & Meta
2238,2025-04,client_9958f0a7ae1df715,content_d02be57d816cf3d7,9416.0,45.0,4.201826,0.004779,3-5,Very High,0.018284,0.013505,1,0.123574,CTR_GAP,Improve Title & Meta


Looking at the top 20 entries, we say that they all have the action, Improve Title & Meta due to the fact that all 20 entries need review based on the label. The reason code is the CTR_GAP between the expected CTR and is acutal CTR, also most entires have the Very High impression with a significant CTR gap.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The picks that might be wrong are the page with 349 impressions it wouldnt need review since it doesnt reach the threshold of number of impressions despite confirming the CTR_GAP, so this type of phenomenon could ocurr with other pages.

There are no leakeages since it uses the inforamtion available of only the month of April and it uses features that may tell future performance or future-window information, tnis means that the needs_review label is generated directly from the rule itself.

In [14]:
csv = pd.read_csv("baseline_action_score.csv")
csv.head(20)

,month,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,position_bucket,impression_bucket,expected_ctr,ctr_gap,needs_review,score,reason_code,action
0,2025-04,client_73cda7b4e4f265ea,content_0b4178c50db57c23,23890.0,90.0,2.798314,0.003767,1-3,Very High,0.019535,0.015768,1,0.158959,CTR_GAP,Improve Title & Meta
1,2025-04,client_73cda7b4e4f265ea,content_8989056a93f310f9,3664.0,2.0,4.739496,0.000546,3-5,Very High,0.018284,0.017738,1,0.145570,CTR_GAP,Improve Title & Meta
2,2025-04,client_9958f0a7ae1df715,content_79bfcf05bc81bf82,12394.0,42.0,3.355194,0.003389,3-5,Very High,0.018284,0.014895,1,0.140389,CTR_GAP,Improve Title & Meta
3,2025-04,client_73cda7b4e4f265ea,content_c8a344d6fbdbb873,6432.0,18.0,4.834496,0.002799,3-5,Very High,0.018284,0.015485,1,0.135795,CTR_GAP,Improve Title & Meta
4,2025-04,client_9958f0a7ae1df715,content_f3a75d8cf58dd50b,3869.0,8.0,3.267181,0.002068,3-5,Very High,0.018284,0.016216,1,0.133963,CTR_GAP,Improve Title & Meta
5,2025-04,client_73cda7b4e4f265ea,content_f08cd9083f18f777,2446.0,3.0,4.706492,0.001226,3-5,Very High,0.018284,0.017058,1,0.133093,CTR_GAP,Improve Title & Meta
6,2025-04,client_73cda7b4e4f265ea,content_3c1f61c9fa1aa75b,3448.0,7.0,4.501050,0.002030,3-5,Very High,0.018284,0.016254,1,0.132401,CTR_GAP,Improve Title & Meta
7,2025-04,client_9958f0a7ae1df715,content_43e2f39f8ecdf102,3257.0,7.0,4.426595,0.002149,3-5,Very High,0.018284,0.016135,1,0.130512,CTR_GAP,Improve Title & Meta
8,2025-04,client_73cda7b4e4f265ea,content_2915a57c1fd91871,4877.0,18.0,4.851624,0.003691,3-5,Very High,0.018284,0.014593,1,0.123933,CTR_GAP,Improve Title & Meta
9,2025-04,client_9958f0a7ae1df715,content_d02be57d816cf3d7,9416.0,45.0,4.201826,0.004779,3-5,Very High,0.018284,0.013505,1,0.123574,CTR_GAP,Improve Title & Meta


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.